# Tutorial 8: Visualization and Analysis of Perturbation Embeddings

Once you have computed perturbation embeddings with `embpy`, the next
step is to **explore, compare, and interpret** them. This tutorial walks
through the full analysis and plotting API provided by the `embpy.tl`
(tools) and `embpy.pl` (plotting) modules.

We will cover:

| # | Topic | Why it is useful |
|---|-------|------------------|
| 1 | **Synthetic dataset** | Build a realistic toy AnnData to follow along |
| 2 | **UMAP / t-SNE** | See the global landscape of your perturbation space |
| 3 | **All embeddings grid** | Quickly compare multiple embedding models side-by-side |
| 4 | **Similarity heatmap** | Spot clusters of related perturbations |
| 5 | **Distance heatmap** | Quantify dissimilarity (Euclidean, cosine, Wasserstein) |
| 6 | **Correlation matrix** | Compare Pearson / Spearman relationships |
| 7 | **Cross-embedding correlation** | Measure agreement between two different models |
| 8 | **KNN overlap** | Test if two embedding spaces agree on local neighborhoods |
| 9 | **Perturbation ranking** | Find the most similar perturbations to a query |
| 10 | **Hierarchical clustering (dendrogram)** | Visualise hierarchical relationships |
| 11 | **Leiden clustering** | Discover communities in the embedding space |
| 12 | **Cluster composition** | Understand what makes up each cluster |
| 13 | **Leiden overview (multi-panel)** | One-liner for the full clustering story |
| 14 | **Embedding distributions** | Check the value ranges across dimensions |
| 15 | **Embedding norms** | Compare the scale / magnitude across models |

**Requirements**:
```bash
pip install 'embpy[scanpy]'
```

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from anndata import AnnData

import embpy.tl as tl
import embpy.pl as pl

## 1. Create a Synthetic Dataset

In a real workflow, embeddings are produced by `BioEmbedder` and stored
in `adata.obsm`. Here we simulate two embedding spaces for 50
perturbations (25 genes + 25 drugs) so that every plot runs instantly.

We simulate:
- `X_model_A` (64-dim): e.g. a protein language model embedding for genes,
  a chemical transformer for drugs.
- `X_model_B` (128-dim): e.g. a different model or fingerprint.

In [ ]:
rng = np.random.default_rng(42)
n_obs = 50

# Generate two groups with slightly different distributions
# to make the plots more interesting
gene_emb_a = rng.standard_normal((25, 64)) + 0.5
drug_emb_a = rng.standard_normal((25, 64)) - 0.5
emb_a = np.vstack([gene_emb_a, drug_emb_a]).astype(np.float32)

gene_emb_b = rng.standard_normal((25, 128)) + 0.3
drug_emb_b = rng.standard_normal((25, 128)) - 0.3
emb_b = np.vstack([gene_emb_b, drug_emb_b]).astype(np.float32)

obs = pd.DataFrame(
    {
        "identifier": [f"GENE_{i}" for i in range(25)] + [f"DRUG_{i}" for i in range(25)],
        "perturbation_type": ["genetic"] * 25 + ["drug"] * 25,
        "mechanism": rng.choice(["kinase_inh", "epigenetic", "receptor_ag", "unknown"], size=n_obs),
    },
    index=[f"obs_{i}" for i in range(n_obs)],
)

adata = AnnData(obs=obs)
adata.obsm["X_model_A"] = emb_a
adata.obsm["X_model_B"] = emb_b

print(f"AnnData: {adata.n_obs} observations")
print(f"Embeddings: {list(adata.obsm.keys())}")
print(f"Perturbation types: {adata.obs['perturbation_type'].value_counts().to_dict()}")

---
## 2. UMAP and t-SNE: Visualising the Global Landscape

**Why?** High-dimensional embeddings are hard to reason about.
UMAP and t-SNE project them to 2-D so you can see:
- Whether genetic and drug perturbations form separate regions.
- Whether perturbations with the same mechanism cluster together.
- Potential outliers.

`embpy.pl.plot_embedding_space` computes coordinates on the fly if they
don't exist yet, so you can go straight from embeddings to a figure in
one call.

In [ ]:
# UMAP colored by perturbation type
fig = pl.plot_embedding_space(
    adata,
    obsm_key="X_model_A",
    color="perturbation_type",
    method="umap",
    title="Model A — UMAP by Perturbation Type",
)

In [ ]:
# t-SNE colored by mechanism
fig = pl.plot_embedding_space(
    adata,
    obsm_key="X_model_A",
    color="mechanism",
    method="tsne",
    title="Model A — t-SNE by Mechanism",
)

---
## 3. All Embeddings Grid

**Why?** When you have generated embeddings from multiple models
(e.g. ESM2 vs ProtT5 for genes, ChemBERTa vs MolFormer vs RDKit for
drugs), you want to compare them side-by-side in a single figure.

`embpy.pl.all_embeddings` auto-discovers every embedding key in
`.obsm`, computes UMAP (or t-SNE) for each, and lays them out
in a grid.

In [ ]:
fig = pl.all_embeddings(
    adata,
    method="umap",
    color="perturbation_type",
    ncols=2,
    figsize_per_panel=(6, 5),
)

---
## 4. Similarity Heatmap

**Why?** A similarity matrix (e.g. cosine similarity) reveals:
- Blocks of highly similar perturbations — these may share biological
  mechanisms or chemical scaffolds.
- The overall structure of the embedding space.

Red/warm colours indicate high similarity; blue/cool indicates low
similarity or dissimilarity.

In [ ]:
fig = pl.plot_similarity_heatmap(
    adata=adata,
    obsm_key="X_model_A",
    metric="cosine",
    title="Cosine Similarity — Model A",
    figsize=(10, 8),
)

---
## 5. Distance Heatmap

**Why?** While similarity tells you *how alike* perturbations are,
distance tells you *how far apart* they are in the embedding space.

`embpy` supports three metrics:
- **Euclidean**: standard geometric distance, sensitive to magnitude.
- **Cosine**: 1 - cosine_similarity, orientation-based.
- **Wasserstein** (Earth Mover's Distance): treats each embedding
  vector as a 1-D distribution and measures the cost of transforming
  one into the other. Useful when embedding dimensions have different
  scales.

In [ ]:
fig = pl.distance_heatmap(
    adata,
    obsm_key="X_model_A",
    metric="euclidean",
    figsize=(10, 8),
)

In [ ]:
# Compare with Wasserstein distance
# (slower for large datasets; here we use a subset)
adata_small = adata[:20].copy()
fig = pl.distance_heatmap(
    adata_small,
    obsm_key="X_model_A",
    metric="wasserstein",
    title="Wasserstein Distance — Model A (subset)",
    figsize=(8, 7),
)

---
## 6. Correlation Matrix

**Why?** Pearson or Spearman correlation between observations captures
linear (or monotonic) relationships in the embedding vectors. This is
a common metric in computational biology for comparing gene expression
profiles, and it naturally extends to perturbation embeddings.

- **Pearson**: assumes a linear relationship between embedding dimensions.
- **Spearman**: rank-based, robust to non-linear monotonic relationships.

In [ ]:
fig = pl.correlation_matrix(
    adata,
    obsm_key="X_model_A",
    method="pearson",
    title="Pearson Correlation — Model A",
)

---
## 7. Cross-Embedding Correlation

**Why?** When you have embeddings from two different models, you want
to know: *do the two models agree on which perturbations are similar?*

This function computes pairwise cosine similarities in each embedding
space, then plots one against the other. A high Pearson/Spearman
correlation means both models capture similar structure. A low
correlation reveals that the models are encoding different aspects of
the perturbation.

In [ ]:
fig = pl.cross_embedding_correlation(
    adata,
    obsm_key_a="X_model_A",
    obsm_key_b="X_model_B",
    method="pearson",
)

---
## 8. KNN Overlap (Jaccard)

**Why?** Cross-embedding correlation is a global metric. KNN overlap
measures **local** agreement: for each perturbation, do the *k*-nearest
neighbors in model A overlap with the *k*-nearest neighbors in model B?

This is reported as a Jaccard index (0 = no overlap, 1 = identical
neighborhoods). A high mean Jaccard means both models agree on the
local structure of the embedding space.

The heatmap shows pairwise mean Jaccard for all embedding pairs.

In [ ]:
fig = pl.knn_overlap(
    adata,
    k=10,
    figsize=(6, 5),
)

In [ ]:
# You can also get the raw per-observation Jaccard scores
jaccard_scores, mean_jaccard = tl.compute_knn_overlap(
    adata,
    obsm_key_a="X_model_A",
    obsm_key_b="X_model_B",
    k=10,
)
print(f"Mean Jaccard: {mean_jaccard:.4f}")
print(f"Per-observation Jaccard (first 5): {jaccard_scores[:5]}")

---
## 9. Perturbation Ranking

**Why?** Given a perturbation of interest (e.g. a drug you are
studying), you want to know which other perturbations have the most
similar embedding. Use cases:

- **Drug repurposing**: find drugs with similar chemical features.
- **Gene function prediction**: find genes with similar embeddings,
  suggesting shared function or pathway membership.
- **Quality control**: verify that known related perturbations
  (e.g. drugs in the same MOA class) appear near the top.

In [ ]:
# Rank by name
rankings = tl.rank_perturbations(
    adata, query="obs_0", obsm_key="X_model_A", top_k=10
)
print("Top 10 perturbations similar to obs_0:")
for name, score in rankings:
    print(f"  {name:>10s}  {score:.4f}")

In [ ]:
# Visualise as a bar chart
fig = pl.plot_perturbation_ranking(
    adata=adata,
    query="obs_0",
    obsm_key="X_model_A",
    query_label="GENE_0",
    top_k=15,
)

---
## 10. Hierarchical Clustering (Dendrogram)

**Why?** Dendrograms reveal the hierarchical structure of your
perturbation space. They are useful for:
- Identifying groups of perturbations that are consistently close
  across the tree.
- Deciding on a suitable number of clusters by cutting the tree at
  different heights.
- Comparing with known classifications (e.g. drug MOA classes).

In [ ]:
fig = pl.dendrogram(
    adata,
    obsm_key="X_model_A",
    metric="cosine",
    linkage_method="average",
    figsize=(14, 5),
)

---
## 11. Leiden Clustering

**Why?** Leiden is a community-detection algorithm that partitions
perturbations into groups based on a shared nearest-neighbor graph.
Unlike k-means, it does not require pre-specifying the number of
clusters — you control granularity via the `resolution` parameter.

This is the standard clustering approach in single-cell genomics and
naturally extends to perturbation embeddings.

In [ ]:
tl.leiden(
    adata,
    obsm_key="X_model_A",
    resolution=0.5,
    key_added="leiden_A",
)

print(f"Number of clusters: {adata.obs['leiden_A'].nunique()}")
print(adata.obs['leiden_A'].value_counts())

In [ ]:
# Visualise clusters on UMAP
fig = pl.plot_embedding_space(
    adata,
    obsm_key="X_model_A",
    color="leiden_A",
    method="umap",
    title="UMAP — Leiden Clusters (Model A)",
)

---
## 12. Cluster Composition

**Why?** After clustering, you want to know what each cluster is made
of. Do certain clusters contain mostly genetic perturbations? Do
specific mechanisms concentrate in a single cluster?

This stacked bar chart breaks each cluster down by a metadata column.

In [ ]:
fig = pl.plot_cluster_composition(
    adata,
    cluster_key="leiden_A",
    color_by="perturbation_type",
)

In [ ]:
# Also break down by mechanism
fig = pl.plot_cluster_composition(
    adata,
    cluster_key="leiden_A",
    color_by="mechanism",
)

---
## 13. Leiden Overview (Multi-Panel)

**Why?** `leiden_overview` is a convenience function that runs Leiden
clustering and produces a multi-panel figure in a single call:

1. **UMAP coloured by cluster** — see the spatial layout.
2. **UMAP coloured by metadata** — overlay biological annotations.
3. **Cluster composition** — stacked bar chart.
4. **Cluster sizes** — bar chart of how many perturbations per cluster.

You can select which panels to show via the `plots` parameter.

In [ ]:
fig = pl.leiden_overview(
    adata,
    obsm_key="X_model_A",
    resolution=0.5,
    color_by="perturbation_type",
)

In [ ]:
# Show only selected panels
fig = pl.leiden_overview(
    adata,
    obsm_key="X_model_A",
    resolution=0.8,
    color_by="mechanism",
    plots=["umap_cluster", "composition"],
)

---
## 14. Embedding Distributions

**Why?** Different embedding models produce vectors with very different
value distributions. For example, a transformer may output near-zero
Gaussian values while a binary fingerprint has 0/1 values.

Visualising the distribution of the first few dimensions helps you:
- Decide whether to normalise or scale before downstream analysis.
- Detect dead dimensions (constant across all perturbations).
- Understand the encoding characteristics of each model.

In [ ]:
fig = pl.embedding_distributions(
    adata,
    n_dims=8,
)

---
## 15. Embedding Norms

**Why?** The L2 norm (magnitude) of embedding vectors varies across
models. Before combining or comparing embeddings from different models,
you should check whether their scales are compatible. If one model
produces vectors with norm ~100 and another ~1, distance-based metrics
will be dominated by the larger-norm model.

This box plot provides a quick visual check.

In [ ]:
fig = pl.embedding_norms(adata)

---
## Summary

The `embpy.tl` and `embpy.pl` modules give you a complete toolkit for
embedding analysis:

| Function | Module | What it does |
|----------|--------|--------------|
| `compute_similarity` | `tl` | Pairwise cosine / Pearson / Spearman similarity |
| `compute_distance_matrix` | `tl` | Pairwise Euclidean / cosine / Wasserstein distance |
| `compute_knn_overlap` | `tl` | Jaccard overlap of KNN sets between two embeddings |
| `rank_perturbations` | `tl` | Rank perturbations by similarity to a query |
| `find_nearest_neighbors` | `tl` | Build a scanpy neighbor graph |
| `compute_umap` / `compute_tsne` | `tl` | Dimensionality reduction to 2-D |
| `leiden` / `cluster_embeddings` | `tl` | Leiden / KMeans / Spectral clustering |
| `plot_embedding_space` | `pl` | 2-D scatter (UMAP / t-SNE) |
| `all_embeddings` | `pl` | Grid of 2-D scatter for all embeddings |
| `plot_similarity_heatmap` | `pl` | Similarity matrix heatmap |
| `distance_heatmap` | `pl` | Distance matrix heatmap |
| `correlation_matrix` | `pl` | Pearson / Spearman correlation heatmap |
| `cross_embedding_correlation` | `pl` | Scatter comparing two embedding spaces |
| `knn_overlap` | `pl` | KNN Jaccard heatmap across embeddings |
| `plot_perturbation_ranking` | `pl` | Bar chart of top-k similar perturbations |
| `dendrogram` | `pl` | Hierarchical clustering dendrogram |
| `plot_cluster_composition` | `pl` | Stacked bar of cluster composition |
| `leiden_overview` | `pl` | Multi-panel Leiden clustering figure |
| `embedding_distributions` | `pl` | Violin plots of embedding dimension values |
| `embedding_norms` | `pl` | Box plot of L2 norms across models |

All of these operate on standard `AnnData` objects with embeddings in
`.obsm`, so they integrate seamlessly with the rest of the `embpy`
workflow and the broader scanpy ecosystem.